# Module 4 Learning Activity 1  
## Working with DataFrames and Built-in Functions in PySpark

In this tutorial-style notebook, we will use **PySpark** to analyse the **City of Chicago Traffic Crash Dataset**.

By the end of this activity, you should be able to:

1. Download shared CSV files into Google Colab using `gdown`.
2. Create a `SparkSession`.
3. Load CSV files into Spark DataFrames.
4. Inspect DataFrame structure and preview records.
5. Use PySpark DataFrame operations such as `select`, `filter`, `groupBy`, `count`, `orderBy`, and `limit`.
6. Use built-in PySpark functions such as `col`, `when`, `to_date`, `substring`, `month`, and `date_format`.
7. Answer the required learning activity questions.

**Dataset files used in this notebook**

- Traffic Crashes — Crashes
- Traffic Crashes — Vehicles
- Traffic Crashes — People

> In this class version, the facilitator has already uploaded the datasets to Google Drive. Students will use `gdown` to download the files into their own Colab runtime.


## 1. Install required packages

We need two Python packages:

| Package | Purpose |
|---|---|
| `pyspark` | Allows us to use Apache Spark from Python. |
| `gdown` | Downloads files from shared Google Drive links into Colab. |

The command below uses `pip`, Python's package installer.

**Argument explanation**

- `-q` means **quiet mode**. It hides most installation messages so the notebook output is shorter.


In [17]:
!pip install -q pyspark gdown



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Download the dataset using `gdown`

The next cell downloads the three CSV files from the facilitator's shared Google Drive links.

**Command explanation**

```bash
gdown --fuzzy "Google Drive shared link" -O output_file.csv
```

| Part | Meaning |
|---|---|
| `gdown` | Runs the Google Drive download tool. |
| `--fuzzy` | Allows `gdown` to understand normal Google Drive sharing links, not just file IDs. |
| `"..."` | The shared Google Drive file link. Quotation marks are used because URLs often contain special characters. |
| `-O` | Stands for **output**. It lets us choose the downloaded file name. |
| `Traffic_Crashes_-_Crashes.csv` | The local file name saved inside the Colab session. |

> These files are saved to the temporary Colab runtime. If the runtime restarts, run this download cell again.


In [18]:
# Download the three shared CSV files from Google Drive.
# --fuzzy allows gdown to read normal Google Drive share links.
# -O specifies the output filename to save in the Colab runtime.

!gdown --fuzzy "https://drive.google.com/file/d/1us-LCihIREQzJezBglRG33va-Z8T5jxm/view?usp=drive_link" -O Traffic_Crashes_-_Crashes.csv
!gdown --fuzzy "https://drive.google.com/file/d/1DkNKmlBC5uWVFTlQCaWeyyOHJuzvl_0y/view?usp=drive_link" -O Traffic_Crashes_-_Vehicles.csv
!gdown --fuzzy "https://drive.google.com/file/d/13aP-ZcoLZf7eX76kzLnBP7fqk93hDBMS/view?usp=drive_link" -O Traffic_Crashes_-_People.csv


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy
usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--json] [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --fuzzy


## 3. Check that the files were downloaded

Before loading data into Spark, always confirm that the files exist.

**Command explanation**

- `ls` lists files in the current folder.
- `-l` shows file details such as size.
- `-h` shows file sizes in a human-readable format such as MB or GB.
- `*.csv` means "show all files ending with `.csv`".


In [19]:
!ls -lh *.csv


zsh:1: no matches found: *.csv


## 4. Create a SparkSession

A `SparkSession` is the main entry point for using Spark DataFrames and Spark SQL.

Think of it as the object that connects your Python code to the Spark engine.

**Code explanation**

```python
spark = SparkSession.builder.appName("TrafficCrashAnalysis").getOrCreate()
```

| Part | Meaning |
|---|---|
| `SparkSession` | The class used to start or connect to Spark. |
| `.builder` | Creates a builder object for configuring the Spark session. |
| `.appName("TrafficCrashAnalysis")` | Gives the Spark application a readable name. |
| `.getOrCreate()` | Gets an existing Spark session if one exists; otherwise creates a new one. |


In [20]:
import os

# Set JAVA_HOME to point to the newly installed Homebrew OpenJDK 17
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

# Update the system PATH variable so the java binary can be found
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Verify it works (should print something starting with /opt/homebrew...)
print("Current JAVA_HOME:", os.environ.get("JAVA_HOME"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TrafficCrashAnalysis").getOrCreate()

Current JAVA_HOME: /opt/homebrew/opt/openjdk@17


## 5. Load CSV files into Spark DataFrames

A **DataFrame** is a distributed table-like data structure. It has rows and columns, similar to an Excel table or a SQL table, but it can be processed across multiple machines.

We use `spark.read` to create a `DataFrameReader`, then provide CSV reading options.

**Argument explanation**

| Option | Meaning |
|---|---|
| `.option("header", "true")` | The first row of the CSV file contains column names. |
| `.option("inferSchema", "true")` | Spark tries to detect column data types automatically, such as integer, double, or timestamp. |
| `.csv("file.csv")` | Reads the specified CSV file and returns a Spark DataFrame. |

> `inferSchema=True` is convenient for teaching, but it can be slower for very large files because Spark needs to scan the data to guess data types.


In [21]:
crashes = spark.read.option("header", "true").option("inferSchema", "true").csv("Traffic_Crashes_-_Crashes.csv")
vehicles = spark.read.option("header", "true").option("inferSchema", "true").csv("Traffic_Crashes_-_Vehicles.csv")
people = spark.read.option("header", "true").option("inferSchema", "true").csv("Traffic_Crashes_-_People.csv")


26/06/23 10:17:32 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: Traffic_Crashes_-_Crashes.csv.
java.io.FileNotFoundException: File Traffic_Crashes_-_Crashes.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catal

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/luisfaria/Desktop/sEngineer/masters_SWEAI/2026-T2/BDA/modules/module-04-apache-spark/Traffic_Crashes_-_Crashes.csv. SQLSTATE: 42K03

## 6. Confirm the DataFrame objects

The next cell checks that each variable is a PySpark DataFrame.

**Code explanation**

- `type(crashes)` shows the Python object type.
- If loading worked, you should see a PySpark DataFrame type.


In [ ]:
print(type(crashes))
print(type(vehicles))
print(type(people))


## 7. Preview the data

Use `.show()` to display rows from a Spark DataFrame.

**Argument explanation**

```python
crashes.show(5, truncate=False)
```

| Argument | Meaning |
|---|---|
| `5` | Show the first 5 rows. |
| `truncate=False` | Do not shorten long text values. This is useful when you want to inspect full values. |

> For large datasets, `.show()` only displays a small sample; it does not print the whole dataset.


In [ ]:
crashes.show(5, truncate=False)


In [ ]:
vehicles.show(5, truncate=False)


In [ ]:
people.show(5, truncate=False)


## 8. Inspect schemas

A schema tells us each column name and its data type.

Use this step before writing analysis code so you know which columns are available.

**Code explanation**

- `printSchema()` prints the column names, data types, and whether missing values are allowed.


In [ ]:
crashes.printSchema()


In [ ]:
people.printSchema()


## Q1. Load the data from the CSV files into DataFrames

This has been completed above. We now have three Spark DataFrames:

| DataFrame | Dataset |
|---|---|
| `crashes` | Traffic crash-level information. |
| `vehicles` | Vehicle-level information. |
| `people` | Person-level information. |

A crash may have multiple people and multiple vehicles. These tables are connected by `CRASH_RECORD_ID`.


## Q2. Ratio of crashes with cell phone use to crashes without cell phone use

The `people` table contains a column called `DRIVER_ACTION`. Some values indicate phone use, such as:

- `CELL PHONE USE OTHER THAN TEXTING`
- `TEXTING`

To answer the question at the **crash level**, we will:

1. Create a flag called `PHONE_USED` for each person record.
2. Group by `CRASH_RECORD_ID`.
3. Mark a crash as phone-related if **any person record** in that crash indicates phone use.
4. Count phone-related crashes and non-phone-related crashes.
5. Calculate the ratio.

**Key functions used**

| Function / method | Meaning |
|---|---|
| `col("DRIVER_ACTION")` | Refers to the `DRIVER_ACTION` column. |
| `.isin(list)` | Checks whether a column value is inside a list of values. |
| `when(condition, value).otherwise(value)` | Creates conditional logic, similar to `if...else`. |
| `groupBy("CRASH_RECORD_ID")` | Groups records that belong to the same crash. |
| `spark_max("PHONE_USED")` | Returns the maximum flag value in each group. If any record has `1`, the crash is marked as phone-related. |
| `.count()` | Counts the number of rows. |


In [ ]:
from pyspark.sql.functions import col, when, max as spark_max


In [ ]:
# First, inspect the most common driver actions.
people.groupBy("DRIVER_ACTION").count().orderBy("count", ascending=False).show(50, truncate=False)


In [ ]:
# Define the driver action values that we will treat as cell-phone-related.
phone_actions = ["CELL PHONE USE OTHER THAN TEXTING", "TEXTING"]

# Create a person-level flag.
# PHONE_USED = 1 means the person record indicates cell phone use.
# PHONE_USED = 0 means the person record does not indicate cell phone use.
people_phone_flag = people.withColumn(
    "PHONE_USED",
    when(col("DRIVER_ACTION").isin(phone_actions), 1).otherwise(0)
)

# Convert person-level records into crash-level records.
# If any person in a crash used a phone, the crash is counted as phone-related.
crash_phone_flag = (
    people_phone_flag
    .filter(col("CRASH_RECORD_ID").isNotNull())
    .groupBy("CRASH_RECORD_ID")
    .agg(spark_max("PHONE_USED").alias("PHONE_USED"))
)

phone_crash_count = crash_phone_flag.filter(col("PHONE_USED") == 1).count()
non_phone_crash_count = crash_phone_flag.filter(col("PHONE_USED") == 0).count()

ratio = phone_crash_count / non_phone_crash_count

print("Phone-related crashes:", phone_crash_count)
print("Non-phone-related crashes:", non_phone_crash_count)
print("Ratio:", ratio)
print("Percentage:", ratio * 100, "%")


### How to interpret Q2

If the ratio is `0.002`, it means there are about **2 phone-related crashes for every 1,000 non-phone-related crashes** in the dataset.

Be careful: this analysis depends on how driver actions were recorded. Some phone use may be unknown or unreported.


## Q3. Three age groups involved with the highest number of crashes

The `people` table contains the `AGE` column. We will create a new column called `AGE_GROUP`.

**Key functions used**

| Function / method | Meaning |
|---|---|
| `.cast("integer")` | Converts the age column to integer type. |
| `.withColumn("AGE_GROUP", ...)` | Creates or replaces a column. |
| `when(...).when(...).otherwise(...)` | Creates multiple conditional categories. |
| `countDistinct("CRASH_RECORD_ID")` | Counts unique crashes rather than total person records. |
| `desc("number_of_crashes")` | Sorts from highest to lowest. |
| `.limit(3)` | Keeps only the top three rows. |


In [ ]:
from pyspark.sql.functions import countDistinct, desc


In [ ]:
# Convert AGE to integer and create age groups.
people_age = people.withColumn("AGE", col("AGE").cast("integer"))

people_age = people_age.withColumn(
    "AGE_GROUP",
    when(col("AGE") < 18, "Under 18")
    .when((col("AGE") >= 18) & (col("AGE") <= 25), "18-25")
    .when((col("AGE") >= 26) & (col("AGE") <= 35), "26-35")
    .when((col("AGE") >= 36) & (col("AGE") <= 45), "36-45")
    .when((col("AGE") >= 46) & (col("AGE") <= 55), "46-55")
    .when((col("AGE") >= 56) & (col("AGE") <= 65), "56-65")
    .when(col("AGE") > 65, "Over 65")
    .otherwise("Unknown")
)


In [ ]:
# Count unique crashes for each age group.
top_age_groups = (
    people_age
    .filter(col("AGE_GROUP") != "Unknown")
    .groupBy("AGE_GROUP")
    .agg(countDistinct("CRASH_RECORD_ID").alias("number_of_crashes"))
    .orderBy(desc("number_of_crashes"))
    .limit(3)
)

top_age_groups.show()


### How to interpret Q3

The result shows the age groups that appear in the largest number of unique crash records.

This does **not** automatically mean those age groups are riskier. To measure risk, we would also need population size, driving exposure, kilometres travelled, or licence-holder counts.


## Q4. Month of the year with the highest number of crashes

The `crashes` table contains `CRASH_DATE`, which includes both date and time.

We will:

1. Extract the date part from `CRASH_DATE`.
2. Convert it into a Spark date column.
3. Extract the month number and month name.
4. Count crashes by month.
5. Sort from highest to lowest.

**Key functions used**

| Function | Meaning |
|---|---|
| `substring(col("CRASH_DATE"), 1, 10)` | Takes the first 10 characters from the date string, such as `03/25/2020`. |
| `to_date(..., "MM/dd/yyyy")` | Converts a string into a date using month/day/year format. |
| `month(...)` | Extracts the month number, from 1 to 12. |
| `date_format(..., "MMMM")` | Extracts the full month name, such as January or February. |


In [ ]:
from pyspark.sql.functions import to_date, substring, month, date_format


In [ ]:
# Create a date-only column from CRASH_DATE.
crashes_date = crashes.withColumn(
    "CRASH_DATE_ONLY",
    to_date(substring(col("CRASH_DATE"), 1, 10), "MM/dd/yyyy")
)

# Display the original and converted date columns.
crashes_date.select("CRASH_DATE", "CRASH_DATE_ONLY").show(10, truncate=False)


In [ ]:
# Count crashes by month.
crashes_by_month = (
    crashes_date
    .withColumn("MONTH_NUM", month(col("CRASH_DATE_ONLY")))
    .withColumn("MONTH_NAME", date_format(col("CRASH_DATE_ONLY"), "MMMM"))
    .groupBy("MONTH_NUM", "MONTH_NAME")
    .count()
    .orderBy(desc("count"))
)

crashes_by_month.show(12)


In [ ]:
# Month with the highest number of crashes.
crashes_by_month.limit(1).show()


### How to interpret Q4

The top row gives the month with the highest number of crashes in the dataset.

A better real-world analysis might adjust for:

- number of days in each month,
- years covered in the data,
- weather conditions,
- holiday periods,
- traffic volume.


## Q5. Day of the week with the least number of crashes

We use the same date-only column created in Q4.

**Key function used**

```python
date_format(col("CRASH_DATE_ONLY"), "EEEE")
```

| Part | Meaning |
|---|---|
| `date_format(...)` | Converts a date into a formatted string. |
| `"EEEE"` | Returns the full day name, such as Monday or Tuesday. |
| `.orderBy("count")` | Sorts from smallest to largest. |
| `.limit(1)` | Keeps only the first row after sorting, which is the least frequent day. |


In [ ]:
# Count crashes by day of the week.
crashes_by_day = (
    crashes_date
    .withColumn("DAY_OF_WEEK", date_format(col("CRASH_DATE_ONLY"), "EEEE"))
    .groupBy("DAY_OF_WEEK")
    .count()
    .orderBy("count")
)

crashes_by_day.show()


In [ ]:
# Day of the week with the least number of crashes.
crashes_by_day.limit(1).show()


### How to interpret Q5

The first row shows the day of the week with the lowest number of crashes.

Again, interpretation requires care. A lower count might reflect lower traffic volume on that day rather than safer driving conditions.


## Optional extension: Select only useful columns

Large datasets often have many columns. In real projects, you usually start by selecting the columns needed for analysis.

**Code explanation**

- `.select(...)` keeps only selected columns.
- This can make later analysis easier to read.


In [ ]:
crashes_small = crashes.select("CRASH_RECORD_ID", "CRASH_DATE", "WEATHER_CONDITION", "LIGHTING_CONDITION")
people_small = people.select("CRASH_RECORD_ID", "PERSON_TYPE", "AGE", "DRIVER_ACTION")

crashes_small.show(5, truncate=False)
people_small.show(5, truncate=False)


## Summary of PySpark concepts used

| Concept | Example | Purpose |
|---|---|---|
| Start Spark | `SparkSession.builder.getOrCreate()` | Creates the Spark entry point. |
| Load CSV | `spark.read.option(...).csv(...)` | Reads CSV files into DataFrames. |
| Preview data | `.show(5)` | Displays sample rows. |
| Inspect schema | `.printSchema()` | Shows column names and data types. |
| Refer to column | `col("AGE")` | Allows column operations. |
| Filter rows | `.filter(condition)` | Keeps rows that match a condition. |
| Add column | `.withColumn("new_col", expression)` | Creates or replaces a column. |
| Group data | `.groupBy("column")` | Groups rows by values. |
| Aggregate | `.count()`, `.agg(...)` | Calculates summary values. |
| Sort results | `.orderBy(desc("count"))` | Orders output rows. |
| Keep top rows | `.limit(3)` | Returns only the first rows after sorting. |


## Discussion forum post

Use the **PySpark DataFrame and Functions** discussion forum to discuss:

- Any installation or dataset download issues.
- Any error messages you encountered when running PySpark code.
- Your interpretation of the results for phone use, age groups, month, and day of week.
- Any suggestions you have for improving the analysis.

When replying to another student, try to include:

1. What you think caused the issue or result.
2. A possible solution or improvement.
3. A short explanation using PySpark terminology.


## Optional: Stop the SparkSession

Run this cell when you have finished the activity.

**Code explanation**

- `spark.stop()` shuts down the Spark session and releases resources.
- In Colab, this is optional, but it is a good habit in Spark projects.


In [ ]:
spark.stop()
